In [ ]:
import os
import sys

# Move up one directory to the project root and add it to the Python path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Now your imports will work perfectly!
from src.data_loader import load_insurance_data, calculate_insurance_metrics
from src.eda_utils import get_missing_summary, plot_loss_ratio_by_dimension

In [ ]:
# Execution Cell Sequence
import pandas as pd
import shap
import matplotlib.pyplot as plt
from src.data_loader import load_insurance_data
from src.modeling import prepare_insurance_features, train_and_evaluate_regression

# Load data assets
zip_path = '../data/MachineLearningRating_v3.zip' 
df = load_insurance_data(zip_path)

# Split data and target vectors
X_train, X_test, y_train, y_test = prepare_insurance_features(df, is_regression=True)

# Fit models and return comparative tables
metrics, best_xgb_model = train_and_evaluate_regression(X_train, X_test, y_train, y_test)
print(pd.DataFrame(metrics).T.to_markdown())


3. Model Interpretability with SHAP
To generate explainable outputs for management review, run the SHAP calculation sequence against your best-performing ensemble tree:

In [ ]:
# Calculate feature attributions using TreeSHAP
explainer = shap.TreeExplainer(best_xgb_model)
shap_values = explainer(X_test)

# Generate Summary Plot 
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, show=False)
plt.title("Risk Driver Feature Importance (SHAP values)", fontsize=14)
plt.tight_layout()
plt.savefig("../reports/shap_importance_summary.png")
plt.show()


Explaining SHAP Values to Leadership
When interpreting your SHAP plots, provide clear business context for the top features:
•	CustomValueEstimate: High asset values pull SHAP points strongly to the right. This indicates that more valuable vehicles significantly increase potential claim severity, requiring higher baseline premium levels.
•	RegistrationYear (Vehicle Age): If older vehicles shift SHAP boundaries towards higher costs, it indicates that aging vehicle components increase repair complexity, which should be accounted for using an age-based premium scalar.


Python Code: Modeling & Premium Optimization

Task 4: Statistical Modeling & Premium OptimizationModeling GoalsPredictive Modeling: Accurately forecast Claim Frequency and Claim Severity based on historical risk variables.Premium Optimization: Translate risk forecasts into fair, data-driven, risk-adjusted commercial premiums.Machine Learning FrameworkWe deploy three models to balance interpretability, variance control, and raw predictive power:Linear Regression: Serves as the parametric statistical baseline.Random Forest Regressor: A non-linear ensemble method that handles categorical boundaries and feature interactions seamlessly.XGBoost: A gradient-boosting architecture designed to minimize residual errors and robustly handle skewed insurance target distributions.Premium Optimization FormulaOnce expected risk metrics are predicted, commercial premiums are calculated dynamically:$$Premium = Pure\ Premium + Expenses + Risk\ Margin + Profit\ Margin$$Where:$$Pure\ Premium = Predicted\ Frequency \times Predicted\ Severity$$Expenses: Fixed underwriting administrative costs ($150$).Risk Margin: A $10\%$ safety buffer scaled to the calculated Pure Premium to absorb unexpected claim volatility.Profit Margin: A flat $5\%$ target financial return for the commercial business line.

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

# --- Simulated Modeling Dataset Setup ---
np.random.seed(42)
n_records = 2000
X = pd.DataFrame({
    'Age': np.random.randint(18, 70, n_records),
    'EngineCapacity': np.random.uniform(1.0, 4.0, n_records),
    'Is_Urban': np.random.choice([0, 1], n_records)
})

# Generating synthetic target values based on functions of features + noise
y_frequency = 0.05 + 0.002 * X['Age'] + 0.05 * X['EngineCapacity'] + np.random.normal(0, 0.02, n_records)
y_severity = 200 + 10 * X['Age'] + 150 * X['EngineCapacity'] + np.random.normal(0, 50, n_records)

# Enforce positive realistic constraints
y_frequency = np.clip(y_frequency, 0, 2)
y_severity = np.clip(y_severity, 0, None)

# --- Train/Test Splits ---
X_train, X_test, y_freq_train, y_freq_test = train_test_split(X, y_frequency, test_size=0.2, random_state=42)
_, _, y_sev_train, y_sev_test = train_test_split(X, y_severity, test_size=0.2, random_state=42)

# --- 1. Model Training & Evaluation Loop ---
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, learning_rate=0.05, random_state=42)
}

print("--- Model Evaluation Metrics (Severity Predictors Example) ---")
for name, model in models.items():
    model.fit(X_train, y_sev_train)
    preds = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_sev_test, preds))
    r2 = r2_score(y_sev_test, preds)
    print(f"{name} -> RMSE: {rmse:.2f} | R² Score: {r2:.4f}")

# --- 2. Production Optimization Engine (Using XGBoost as standard Champion Model) ---
print("\n--- Generating Premium Optimization Structure ---")
xgb_freq = XGBRegressor(random_state=42).fit(X_train, y_freq_train)
xgb_sev = XGBRegressor(random_state=42).fit(X_train, y_sev_train)

# Predict risk elements on the test pipeline
test_predictions = pd.DataFrame(X_test, columns=X.columns).reset_index(drop=True)
test_predictions['Pred_Frequency'] = xgb_freq.predict(X_test)
test_predictions['Pred_Severity'] = xgb_sev.predict(X_test)

# --- 3. Premium Equation Pipeline ---
# Constants Constants
EXPENSES = 150.00
RISK_MARGIN_PCT = 0.10  # 10% safety buffer
PROFIT_MARGIN_PCT = 0.05 # 5% profit target

# Applying: Pure Premium = Freq * Sev
test_predictions['Pure_Premium'] = test_predictions['Pred_Frequency'] * test_predictions['Pred_Severity']

# Applying: Premium = Pure Premium + Expenses + Risk Margin + Profit Margin
test_predictions['Risk_Margin'] = test_predictions['Pure_Premium'] * RISK_MARGIN_PCT
test_predictions['Profit_Margin'] = test_predictions['Pure_Premium'] * PROFIT_MARGIN_PCT

test_predictions['Final_Optimized_Premium'] = (
    test_predictions['Pure_Premium'] + 
    EXPENSES + 
    test_predictions['Risk_Margin'] + 
    test_predictions['Profit_Margin']
)

# Display a subset sample of the calculated results
print(test_predictions[['Age', 'EngineCapacity', 'Pred_Frequency', 'Pred_Severity', 'Pure_Premium', 'Final_Optimized_Premium']].head())

--- Model Evaluation Metrics (Severity Predictors Example) ---
Linear Regression -> RMSE: 50.53 | R² Score: 0.9423
Random Forest -> RMSE: 55.86 | R² Score: 0.9294
XGBoost -> RMSE: 53.69 | R² Score: 0.9348

--- Generating Premium Optimization Structure ---
   Age  EngineCapacity  Pred_Frequency  Pred_Severity  Pure_Premium  \
0   56        2.538179        0.284868    1145.349121    326.273682   
1   55        2.935870        0.297442    1168.240234    347.483398   
2   66        2.248364        0.307058    1212.401489    372.277924   
3   34        2.576305        0.238504     959.886230    228.936554   
4   40        2.025347        0.231291     860.187195    198.953278   

   Final_Optimized_Premium  
0               525.214722  
1               549.605896  
2               578.119629  
3               413.277039  
4               378.796265  
